In [7]:
import torch
import os
import sys
sys.path.insert(0, './d2l-en/pytorch')
from d2l import torch as d2l

def coor2d(X, K):
    h, w = K.shape  # gets the shape of each Kernel
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    # creates an output tensor Y with the appropriate shape 
    # considering the size of input X and kernel K

    # output is the sum of the element-wise product of the kernel 
    # and the corresponding region in the input
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w] * K).sum()
    return Y


def corr2d_multi_in(X, K):
    # zip used to iterate over multiple input channels and kernels 
    # matches kernel for R with channel R, kernel for G with channel G, and kernel for B with channel B
    # sums up the results from each channel to produce the final output
    return sum(coor2d(x, k) for x, k in zip(X, K))

In [15]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]], [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

calculate multichannel ouptut

In [16]:
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)
# this is to return multiple features map (eyes, edges, etc.) 
# by applying multiple kernels to the same input


K = torch.stack((K, K + 1, K + 2), 0)  
# creates a new tensor by stacking K, K+1, and K+2 along a new dimension (the output channel dimension)
K.shape

torch.Size([3, 2, 2, 2])

In [17]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

1x1 convolution layer

In [19]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    # flattening, collapse 2d dimensions into a single long row
    # e.g. if 3x8x8 -> 3x64
    K = K.reshape((c_o, c_i))
    # since kernel size is 1x1, we can reshape it to have the 
    # same number of rows as output channels and 
    # the same number of columns as input channels
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))
    # unflattening, reshape back to original height and width
    # but with the number of output channels

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1-Y2).sum()) < 1e-6